# Module 2.1: Find a source and add related graph facts

Use this notebook when a question needs facts from both a source document and related graph records. For example, to find Chicago hotels with both a spa and a swimming pool, you need to check more than one fact for each hotel.

Start with semantic search to find the matching source. Then follow graph relationships to return the related facts and show where they came from.

**Overview**

- **Semantic search:** Finds source text with meaning similar to the question.
- **Exact-term search:** Finds source text that contains an important word or identifier.
- **Graph expansion:** Adds related hotel facts, such as amenities and ratings, to a matching source.
- **Structured filtering:** Checks clear conditions, such as whether the same hotel has both required amenities.
- **What is checked:** This notebook checks retrieved records and their sources. It does not evaluate generated answer wording.

The optional Text2Cypher example appears after the required retrieval exercises.

## Check that Module 1 finished

Run the cells in order after you finish Module 1. The next check runs in Python, so you do not need a terminal command.

- **Read-only:** This notebook reads the graph. It does not rebuild or clear your work.
- **If the check fails:** Return to Module 1, run every cell, then restart this notebook from the top.

In [ ]:
import os
import sys
from pathlib import Path


def locate_notebooks_root():
    override = os.environ.get('WORKSHOP_NOTEBOOKS_DIR')
    if override:
        candidate = Path(override).expanduser().resolve()
        if (candidate / 'workshop').is_dir():
            return candidate
        raise RuntimeError('WORKSHOP_NOTEBOOKS_DIR must contain the workshop package')

    starting_path = Path.cwd().resolve()
    for candidate in (starting_path, *starting_path.parents):
        if candidate.name == 'notebooks' and (candidate / 'workshop').is_dir():
            return candidate
        nested = candidate / 'notebooks'
        if (nested / 'workshop').is_dir():
            return nested
    raise RuntimeError(
        'Could not locate notebooks/workshop. Set WORKSHOP_NOTEBOOKS_DIR.'
    )


NOTEBOOKS_ROOT = locate_notebooks_root()
REPO_ROOT = NOTEBOOKS_ROOT.parent
MODULE_DIR = NOTEBOOKS_ROOT / '02-connected-context'
if str(NOTEBOOKS_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_ROOT))
print(f'Workshop root: {REPO_ROOT}')

In [ ]:
from dotenv import load_dotenv
from IPython.display import HTML, display
from neo4j import GraphDatabase, Query, READ_ACCESS
from neo4j_graphrag.retrievers import (
    HybridRetriever,
    VectorCypherRetriever,
    VectorRetriever,
)
from neo4j_graphrag.types import RetrieverResultItem

from neo4j_graphrag.retrievers.text2cypher import extract_cypher

load_dotenv(NOTEBOOKS_ROOT / '.env')
load_dotenv(REPO_ROOT / '.env')
load_dotenv(REPO_ROOT / 'CONFIG.txt')

from workshop.aws_region import aws_region, configure_aws_region
from workshop.bedrock_providers import BedrockEmbeddings, BedrockLLM
from workshop.graph_connection import (
    graph_database,
    neo4j_auth,
    neo4j_uri,
    require_neo4j_env,
)
from workshop.graph_schema import GRAPH_SCHEMA
from workshop.hybrid_retrieval import (
    GRAPH_QUERY_EXAMPLES,
    GRAPH_QUERY_PROMPT,
    pinned_schema_text,
    search_hotel_knowledge,
)
from workshop.retrieval_contract import (
    CHUNK_FULLTEXT_INDEX,
    CHUNK_VECTOR_INDEX,
    EMBEDDING_DIMENSIONS,
)
from workshop.retrieval_setup import (
    CHICAGO_CITY,
    CHICAGO_EXCLUSION,
    CHICAGO_FILTER_QUERY,
    CHICAGO_QUALIFIER,
    CHICAGO_SOURCE_FILES,
    chicago_filter_problems,
    chicago_filter_records,
    source_fixture_problems,
    verify_retrieval_indexes,
)

configure_aws_region()
require_neo4j_env()
DATABASE = graph_database()
driver = GraphDatabase.driver(neo4j_uri(), auth=neo4j_auth())
driver.verify_connectivity()
print(f'Connected to Neo4j database: {DATABASE}')

## Check that the graph has the required data

Run this cell before the retrieval examples. It checks that the Cairo and Chicago source documents, embeddings, entities, relationships, and indexes are ready.

In [ ]:
try:
    verify_retrieval_indexes(driver)
    problems = source_fixture_problems(driver)
    chicago_records = chicago_filter_records(driver)
    problems.extend(chicago_filter_problems(chicago_records))
    if problems:
        raise RuntimeError('; '.join(problems))
except Exception as exc:
    raise RuntimeError(
        f'Module 2.1 is not ready: {exc}\n'
        'Return to the Module 1 notebook, run its cells through completion, '
        'and then rerun this notebook from the top. This readiness check did '
        'not modify the graph.'
    ) from exc

print(f'PASS  {CHUNK_VECTOR_INDEX}: online, cosine, {EMBEDDING_DIMENSIONS} dimensions')
print(f'PASS  {CHUNK_FULLTEXT_INDEX}: online over Chunk.text')
print('PASS  Cairo and Chicago source, path, field, amenity, and filter fixtures')

## Show the graph structure and embedding settings

This cell shows the graph structure used by the extraction pipeline and each graph query.

- **`GRAPH_SCHEMA`:** The shared list of node labels and relationships used in this workshop.
- **Embedding settings:** Each text retriever uses the same 1024-dimensional Amazon Nova settings that created the chunk vectors.
- **Database:** Each query uses the configured Neo4j database.

In [ ]:
pattern_rows = ''.join(
    f'<tr><td><strong>{source}</strong></td><td>-[:{relationship}]-&gt;</td>'
    f'<td><strong>{target}</strong></td></tr>'
    for source, relationship, target in GRAPH_SCHEMA['patterns']
)
display(HTML(
    '<table><thead><tr><th>From</th><th>Relationship</th><th>To</th></tr>'
    f'</thead><tbody>{pattern_rows}</tbody></table>'
    '<p>Text provenance: <code>(:Chunk)-[:FROM_DOCUMENT]-&gt;(:Document)</code>.</p>'
    '<p>Entity provenance: <code>(:Hotel)-[:FROM_CHUNK]-&gt;(:Chunk)</code>.</p>'
))

## Set up the retrieved-context display

The next code cell defines one shared display format for every retrieval result. Every retriever prints the same fields in the same order, so you can compare results side by side.

- **Question and retriever:** Show what was asked and which retriever ran.
- **Ranking details:** Show the top-k setting, rank, and score.
- **Source text:** Show the source filename and complete `Chunk.text`.
- **Context size:** Show the size of the structured fields and source text separately.
- **Missing fields:** Show any requested fields that the result does not contain.

Each source filename comes from the graph provenance path for that result.

In [ ]:
def source_for_chunk_text(chunk):
    with driver.session(database=DATABASE, default_access_mode=READ_ACCESS) as session:
        record = session.run(
            '''
            MATCH (matched:Chunk {text: $chunk})-[:FROM_DOCUMENT]->(document:Document)
            RETURN collect(DISTINCT document.source_filename) AS source_filenames
            ''',
            chunk=chunk,
        ).single(strict=True)
    source_filenames = record['source_filenames']
    assert len(source_filenames) == 1, (
        f'Retrieved Chunk must resolve to one source; observed {source_filenames}'
    )
    return source_filenames[0]


def context_value_chars(value):
    if value is None:
        return 0
    if isinstance(value, dict):
        return sum(
            context_value_chars(key) + context_value_chars(item)
            for key, item in value.items()
        )
    if isinstance(value, (list, tuple, set)):
        return sum(context_value_chars(item) for item in value)
    return len(str(value))


def context_char_counts(structured_fields, source_text):
    structured_chars = sum(
        context_value_chars(value) for value in structured_fields.values()
    )
    return structured_chars, len(source_text)


def text_result_formatter(record):
    node = record.get('node') or {}
    chunk = node.get('text') or ''
    return RetrieverResultItem(
        content=chunk,
        metadata={
            'score': record.get('score'),
            'source_filename': source_for_chunk_text(chunk),
        },
    )


embedder = BedrockEmbeddings(region_name=aws_region())

def context_rows(question, retriever_name, result, top_k, configuration, field_checks):
    print(f'Question: {question}')
    print(f'Retriever: {retriever_name}')
    print(f'Configuration: {configuration}')
    print(f'Top-k: {top_k}')
    print(f'Result count: {len(result.items)}')
    rows = []
    for rank, item in enumerate(result.items, 1):
        metadata = item.metadata or {}
        chunk = str(item.content or '')
        missing = [
            name for name, check in field_checks.items()
            if not check(chunk, metadata)
        ]
        structured_chars, source_text_chars = context_char_counts(
            {'source_filename': metadata.get('source_filename')}, chunk
        )
        row = {
            'rank': rank,
            'score': metadata.get('score'),
            'source_filename': metadata.get('source_filename'),
            'structured_context_chars': structured_chars,
            'source_text_chars': source_text_chars,
            'missing_requested_fields': missing,
            'chunk': chunk,
        }
        rows.append(row)
        score = row['score']
        score_text = 'n/a' if score is None else f'{score:.6f}'
        print(f'Rank {rank} | score={score_text} | source={row["source_filename"]}')
        print(f'Structured-field context: {structured_chars} characters')
        print(f'Source-text context: {source_text_chars} characters')
        print(f'Missing requested fields: {missing or "none"}')
        print('Complete Chunk text:')
        print(chunk)
        print()
    return rows

## 1. Find an arrival time with semantic search

This example uses `VectorRetriever` to find a source by meaning. The question says "arrival processing" instead of `Standard check-in time`, so it tests whether semantic search can find the Cairo source when the wording changes.

- **Input:** A question about the arrival time at AnyCompany Cairo Nile View.
- **Expected context:** The Cairo source and its supported `3:00 PM` arrival time.
- **Check:** The test checks the retrieved context only.

In [ ]:
vector_retriever = VectorRetriever(
    driver=driver,
    index_name=CHUNK_VECTOR_INDEX,
    embedder=embedder,
    return_properties=['text'],
    result_formatter=text_result_formatter,
    neo4j_database=DATABASE,
)
ARRIVAL_QUESTION = (
    'When does standard arrival processing begin at AnyCompany Cairo Nile View?'
)
VECTOR_TOP_K = 3
arrival_result = vector_retriever.search(
    query_text=ARRIVAL_QUESTION,
    top_k=VECTOR_TOP_K,
)
arrival_rows = context_rows(
    ARRIVAL_QUESTION,
    'VectorRetriever',
    arrival_result,
    VECTOR_TOP_K,
    f'index={CHUNK_VECTOR_INDEX}, cosine, Nova {EMBEDDING_DIMENSIONS} dimensions',
    {
        'source_filename': lambda text, metadata: bool(metadata.get('source_filename')),
        'supported_arrival_time': lambda text, metadata: '3:00 PM' in text,
    },
)
cairo_arrival = [
    row for row in arrival_rows
    if row['source_filename'] == 'hotel-cairo-001.txt'
]
assert cairo_arrival, 'The Cairo source must appear in the top three vector results'
assert '3:00 PM' in cairo_arrival[0]['chunk'], (
    f"Expected Cairo arrival time 3:00 PM; observed {cairo_arrival[0]['chunk']!r}"
)
print('PASS  Cairo source and supported 3:00 PM arrival time are visible.')

## 2. Find a postal code with hybrid search

This example compares `VectorRetriever` and `HybridRetriever` with the same top-five limit. Postal code `60611` is an exact identifier. The full-text index matches its exact characters, while semantic search only matches similar meaning, so full-text search finds it more reliably.

- **Vector signal:** The complete question supplies the semantic search input.
- **Full-text signal:** `60611` supplies the exact-term search input.
- **Hybrid ranking:** The reviewed linear ranker combines both signals with `alpha=0.2`.

In [ ]:
IDENTIFIER_QUESTION = 'What is the cancellation policy for the hotel at 60611?'
IDENTIFIER_TOP_K = 5
vector_identifier_result = vector_retriever.search(
    query_text=IDENTIFIER_QUESTION,
    top_k=IDENTIFIER_TOP_K,
)

hybrid_retriever = HybridRetriever(
    driver=driver,
    vector_index_name=CHUNK_VECTOR_INDEX,
    fulltext_index_name=CHUNK_FULLTEXT_INDEX,
    embedder=embedder,
    return_properties=['text'],
    result_formatter=text_result_formatter,
    neo4j_database=DATABASE,
)
hybrid_identifier_result = hybrid_retriever.search(
    query_text='60611',
    query_vector=vector_identifier_result.metadata['query_vector'],
    top_k=IDENTIFIER_TOP_K,
    ranker='linear',
    alpha=0.2,
)

identifier_checks = {
    'source_filename': lambda text, metadata: bool(metadata.get('source_filename')),
    'postal_code_60611': lambda text, metadata: '60611' in text,
    'cancellation_policy': lambda text, metadata: (
        'at least 24 hours prior to arrival' in text.casefold()
    ),
}
vector_identifier_rows = context_rows(
    IDENTIFIER_QUESTION,
    'VectorRetriever',
    vector_identifier_result,
    IDENTIFIER_TOP_K,
    f'index={CHUNK_VECTOR_INDEX}, semantic signal=complete question',
    identifier_checks,
)
hybrid_identifier_rows = context_rows(
    IDENTIFIER_QUESTION,
    'HybridRetriever',
    hybrid_identifier_result,
    IDENTIFIER_TOP_K,
    'linear ranker, alpha=0.2, vector signal=complete question, full-text term=60611',
    identifier_checks,
)
for label, rows in (('Vector', vector_identifier_rows), ('Hybrid', hybrid_identifier_rows)):
    for row in rows:
        row['exact_term_hits'] = ['60611'] if '60611' in row['chunk'] else []
        print(
            f"{label} rank {row['rank']} | source={row['source_filename']} | "
            f"exact_term_hits={row['exact_term_hits']}"
        )

windward_hybrid = [
    row for row in hybrid_identifier_rows
    if row['source_filename'] == 'hotel-chicago-001.txt'
]
assert windward_hybrid, 'Hybrid must return hotel-chicago-001.txt in its top five'
windward_chunk = windward_hybrid[0]['chunk']
assert 'Windward Mile Tower' in windward_chunk, (
    f'Expected Windward Mile Tower in hybrid context; observed {windward_chunk!r}'
)
assert '60611' in windward_chunk, (
    f'Expected postal code 60611 in hybrid context; observed {windward_chunk!r}'
)
assert 'at least 24 hours prior to arrival' in windward_chunk.casefold(), (
    f'Expected cancellation policy in hybrid context; observed {windward_chunk!r}'
)
vector_rank_scan = vector_retriever.search(
    query_vector=vector_identifier_result.metadata['query_vector'],
    top_k=30,
)
live_vector_rank = next(
    (
        rank
        for rank, item in enumerate(vector_rank_scan.items, 1)
        if (item.metadata or {}).get('source_filename') == 'hotel-chicago-001.txt'
    ),
    None,
)
rank_text = str(live_vector_rank) if live_vector_rank is not None else 'outside top 30'
print(f'Live vector rank for hotel-chicago-001.txt: {rank_text}')
print('PASS  Hybrid context contains the hotel, postal code, and cancellation policy.')

## 3. Find a source, then add related hotel facts

This example uses `VectorCypherRetriever` to combine semantic search with a reviewed graph query. It finds the matching source `Chunk` first, then follows graph relationships to add hotel fields and amenities.

- **Source record:** The result keeps the matching `Chunk`, its `Document`, and the semantic score.
- **Added facts:** The graph query returns the hotel name, hotel ID, guest rating, and amenities.
- **Provenance:** Each field shows the graph path that produced it.
- **Source of truth:** The graph result reflects what extraction placed in Neo4j. It is not an independent source of truth. Compare it with the source text to find missing or merged facts.

In [ ]:
VECTOR_CYPHER_QUERY = '''
MATCH (node)-[:FROM_DOCUMENT]->(document:Document)
OPTIONAL MATCH (hotel:Hotel)-[:FROM_CHUNK]->(node)
OPTIONAL MATCH (hotel)-[:OFFERS_AMENITY]->(amenity:Amenity)
WITH node, score, document, hotel, collect(DISTINCT amenity.name) AS amenities
RETURN hotel.name AS hotel_name,
       hotel.hotel_id AS hotel_id,
       hotel.guest_rating AS guest_rating,
       document.source_filename AS source_filename,
       amenities,
       node.text AS source_chunk,
       score AS semantic_score,
       CASE
           WHEN hotel IS NULL THEN ['FROM_DOCUMENT']
           ELSE ['FROM_DOCUMENT', 'FROM_CHUNK', 'OFFERS_AMENITY']
       END AS relationship_types,
       CASE
           WHEN hotel IS NULL THEN 'missing Hotel enrichment for semantic hit'
           ELSE 'complete Hotel enrichment'
       END AS graph_enrichment_status,
       {
           source_chunk: '(:Chunk)-[:FROM_DOCUMENT]->(:Document)',
           source_filename: '(:Chunk)-[:FROM_DOCUMENT]->(:Document)',
           hotel_name: '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
           hotel_id: '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
           guest_rating: '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
           amenities: '(:Hotel)-[:OFFERS_AMENITY]->(:Amenity)'
       } AS field_provenance
ORDER BY semantic_score DESC,
         CASE WHEN hotel_id IS NULL THEN 1 ELSE 0 END ASC,
         hotel_id ASC
'''

def graph_result_formatter(record):
    requested = (
        'hotel_name',
        'hotel_id',
        'guest_rating',
        'source_filename',
        'amenities',
    )
    metadata = {
        'hotel_name': record.get('hotel_name'),
        'hotel_id': record.get('hotel_id'),
        'guest_rating': record.get('guest_rating'),
        'source_filename': record.get('source_filename'),
        'amenities': record.get('amenities') or [],
        'semantic_score': record.get('semantic_score'),
        'relationship_types': record.get('relationship_types') or [],
        'graph_enrichment_status': record.get('graph_enrichment_status'),
        'field_provenance': record.get('field_provenance') or {},
    }
    metadata['missing_requested_fields'] = [
        field for field in requested
        if metadata.get(field) is None or metadata.get(field) == []
    ]
    return RetrieverResultItem(
        content=record.get('source_chunk') or '',
        metadata=metadata,
    )

vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name=CHUNK_VECTOR_INDEX,
    retrieval_query=VECTOR_CYPHER_QUERY,
    embedder=embedder,
    result_formatter=graph_result_formatter,
    neo4j_database=DATABASE,
)
CAIRO_GRAPH_QUESTION = (
    'What amenities and guest rating does AnyCompany Cairo Nile View have?'
)
GRAPH_TOP_K = 3
graph_vector_result = vector_retriever.search(
    query_text=CAIRO_GRAPH_QUESTION,
    top_k=GRAPH_TOP_K,
)
graph_vector_rows = context_rows(
    CAIRO_GRAPH_QUESTION,
    'VectorRetriever',
    graph_vector_result,
    GRAPH_TOP_K,
    f'index={CHUNK_VECTOR_INDEX}, semantic entry before graph expansion',
    {
        'source_filename': lambda text, metadata: bool(metadata.get('source_filename')),
        'hotel_name': lambda text, metadata: 'AnyCompany Cairo Nile View' in text,
        'hotel_id': lambda text, metadata: False,
        'guest_rating': lambda text, metadata: '4.5/5.0' in text,
        'amenities': lambda text, metadata: 'Hotel Amenities' in text,
    },
)
graph_result = vector_cypher_retriever.search(
    query_text=CAIRO_GRAPH_QUESTION,
    top_k=GRAPH_TOP_K,
)
graph_records = []
print(f'Question: {CAIRO_GRAPH_QUESTION}')
print('Retriever: VectorCypherRetriever')
print(f'Configuration: index={CHUNK_VECTOR_INDEX}, top_k={GRAPH_TOP_K}, reviewed traversal')
print(f'Result count: {len(graph_result.items)}')
for rank, item in enumerate(graph_result.items, 1):
    metadata = dict(item.metadata or {})
    metadata['source_chunk'] = str(item.content or '')
    metadata['rank'] = rank
    structured_context = {
        field: metadata[field]
        for field in (
            'hotel_name',
            'hotel_id',
            'guest_rating',
            'source_filename',
            'amenities',
            'relationship_types',
            'field_provenance',
            'graph_enrichment_status',
        )
    }
    structured_chars, source_text_chars = context_char_counts(
        structured_context, metadata['source_chunk']
    )
    metadata['structured_context_chars'] = structured_chars
    metadata['source_text_chars'] = source_text_chars
    graph_records.append(metadata)
    print(f'Record {rank}:')
    for field in (
        'hotel_name',
        'hotel_id',
        'guest_rating',
        'source_filename',
        'amenities',
        'semantic_score',
        'relationship_types',
        'graph_enrichment_status',
        'field_provenance',
        'missing_requested_fields',
        'structured_context_chars',
        'source_text_chars',
    ):
        print(f'  {field}: {metadata[field]}')
    print('  source_chunk:')
    print(metadata['source_chunk'])

missing_enrichment = [
    record for record in graph_records
    if record['graph_enrichment_status'].startswith('missing')
]
print(
    f'Graph enrichment: {len(graph_records) - len(missing_enrichment)} complete, '
    f'{len(missing_enrichment)} missing Hotel context. Semantic hits without an '
    'extracted Hotel stay visible instead of disappearing.'
)
cairo_matches = [
    record for record in graph_records
    if record['source_filename'] == 'hotel-cairo-001.txt'
]
assert len(cairo_matches) == 1, (
    f'Expected one Cairo graph record; observed {len(cairo_matches)} in {graph_records}'
)
cairo_graph = cairo_matches[0]
assert cairo_graph['hotel_name'] == 'AnyCompany Cairo Nile View', (
    f"Expected Cairo hotel name; observed {cairo_graph['hotel_name']!r}"
)
assert cairo_graph['hotel_id'] == '81393d51-1df3-4f53-b58e-e4cda9736fd7', (
    f"Expected locked Cairo hotel ID; observed {cairo_graph['hotel_id']!r}"
)
assert cairo_graph['guest_rating'] == 4.5, (
    f"Expected Cairo rating 4.5; observed {cairo_graph['guest_rating']!r}"
)
assert not cairo_graph['missing_requested_fields'], (
    f"Expected every requested Cairo field; missing {cairo_graph['missing_requested_fields']}"
)
assert cairo_graph['semantic_score'] is not None, (
    f"Expected a semantic score; observed {cairo_graph['semantic_score']!r}"
)
assert set(cairo_graph['relationship_types']) == {
    'FROM_DOCUMENT',
    'FROM_CHUNK',
    'OFFERS_AMENITY',
}, f"Unexpected Cairo relationship types: {cairo_graph['relationship_types']}"
for field in (
    'source_chunk',
    'source_filename',
    'hotel_name',
    'hotel_id',
    'guest_rating',
    'amenities',
):
    assert field in cairo_graph['field_provenance'], (
        f'Expected provenance for {field}; observed {cairo_graph["field_provenance"]}'
    )
amenity_text = ' '.join(cairo_graph['amenities']).casefold()
for required_term in ('pool', 'spa', 'fitness', 'wifi', 'restaurant'):
    assert required_term in amenity_text, required_term

vector_cairo_matches = [
    row for row in graph_vector_rows
    if row['source_filename'] == 'hotel-cairo-001.txt'
]
assert len(vector_cairo_matches) == 1, (
    f'Expected one Cairo vector record; observed {len(vector_cairo_matches)}'
)
vector_cairo = vector_cairo_matches[0]
requested_graph_fields = {
    'hotel_name',
    'hotel_id',
    'guest_rating',
    'source_filename',
    'amenities',
}
vector_named_fields = {'source_filename'}
graph_named_fields = requested_graph_fields - set(cairo_graph['missing_requested_fields'])
print('Context comparison:')
print(
    f'  Vector: {len(vector_named_fields)}/{len(requested_graph_fields)} named fields, '
    f"{vector_cairo['structured_context_chars']} structured and "
    f"{vector_cairo['source_text_chars']} source-text characters"
)
print(
    f'  Vector-Cypher: {len(graph_named_fields)}/{len(requested_graph_fields)} named fields, '
    f"{cairo_graph['structured_context_chars']} structured and "
    f"{cairo_graph['source_text_chars']} source-text characters"
)
print('Extraction quality limits graph enrichment. Missing extracted relationships stay missing.')
print('PASS  Cairo Vector-Cypher context includes every locked field and provenance path.')

## 4. Find Chicago hotels with both required amenities

This example uses a shared, fixed Cypher query to filter Chicago hotels. The query checks both amenity conditions for the same hotel.

- **Candidates:** Every Chicago hotel checked by the query.
- **Qualifier:** A hotel that has both a spa and a swimming pool.
- **Exclusion:** A hotel that is missing one or both required amenities.

In [ ]:
CHICAGO_QUESTION = 'Which hotels in Chicago offer both a spa and a swimming pool?'
candidate_records = chicago_filter_records(driver)
qualifying_records = [row for row in candidate_records if row['qualifies']]
excluded_records = [row for row in candidate_records if not row['qualifies']]

CHICAGO_PATTERN_NAME = 'Reviewed fixed Cypher: same-hotel spa AND pool filter'
CHICAGO_CONFIGURATION = 'city predicate, reviewed two-amenity AND filter'
CHICAGO_REQUESTED_FIELDS = ('hotel_name', 'guest_rating', 'amenities', 'source_filename')
CHICAGO_PROVENANCE = {
    'source_chunk': '(:Chunk)-[:FROM_DOCUMENT]->(:Document)',
    'source_filename': '(:Chunk)-[:FROM_DOCUMENT]->(:Document)',
    'hotel_name': '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
    'guest_rating': '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
    'amenities': '(:Hotel)-[:OFFERS_AMENITY]->(:Amenity)',
}


def fixed_cypher_context(record):
    record['field_provenance'] = CHICAGO_PROVENANCE
    record['missing_requested_fields'] = [
        field for field in CHICAGO_REQUESTED_FIELDS
        if record.get(field) is None or record.get(field) == []
    ]
    structured_fields = {
        field: record.get(field)
        for field in (*CHICAGO_REQUESTED_FIELDS, 'qualifies', 'missing_required_amenities')
    }
    structured_chars, source_text_chars = context_char_counts(
        structured_fields, record.get('source_chunk') or ''
    )
    record['structured_context_chars'] = structured_chars
    record['source_text_chars'] = source_text_chars
    return record


candidate_records = [fixed_cypher_context(record) for record in candidate_records]
qualifying_records = [row for row in candidate_records if row['qualifies']]
excluded_records = [row for row in candidate_records if not row['qualifies']]

print(f'Question: {CHICAGO_QUESTION}')
print(f'Pattern: {CHICAGO_PATTERN_NAME}')
print(f'Configuration: {CHICAGO_CONFIGURATION}')
print(f'Reviewed Cypher: {CHICAGO_FILTER_QUERY}')
print(f'Parameters: city={CHICAGO_CITY!r}')
print(f'Candidate count: {len(candidate_records)}')
print(f'Qualifier count: {len(qualifying_records)}')
print(f'Exclusion count: {len(excluded_records)}')
print('Candidate records:')
for record in candidate_records:
    for field in (
        'hotel_name', 'guest_rating', 'amenities', 'source_filename',
        'qualifies', 'missing_required_amenities', 'missing_requested_fields',
        'field_provenance', 'structured_context_chars', 'source_text_chars',
    ):
        print(f'  {field}: {record[field]}')
print('Qualifying records:')
for record in qualifying_records:
    print(record)
print('Excluded records:')
for record in excluded_records:
    print(record)

chicago_problems = chicago_filter_problems(candidate_records)
assert not chicago_problems, f'Expected locked Chicago filter results; observed {chicago_problems}'
observed_sources = {row['source_filename'] for row in candidate_records}
assert observed_sources == set(CHICAGO_SOURCE_FILES), (
    f'Expected Chicago sources {set(CHICAGO_SOURCE_FILES)}; observed {observed_sources}'
)
observed_qualifiers = [row['hotel_name'] for row in qualifying_records]
assert observed_qualifiers == [CHICAGO_QUALIFIER], (
    f'Expected qualifier {CHICAGO_QUALIFIER!r}; observed {observed_qualifiers}'
)
assert len(excluded_records) == 1, (
    f'Expected one Chicago exclusion; observed {len(excluded_records)}'
)
assert excluded_records[0]['hotel_name'] == CHICAGO_EXCLUSION, (
    f"Expected exclusion {CHICAGO_EXCLUSION!r}; observed {excluded_records[0]['hotel_name']!r}"
)
assert set(excluded_records[0]['missing_required_amenities']) == {
    'spa',
    'swimming pool',
}, (
    'Expected Windward to name missing spa and swimming pool; observed '
    f"{excluded_records[0]['missing_required_amenities']}"
)
print('PASS  Two candidates, one qualifier, and the Windward exclusion are explicit.')

## Optional: Generate Cypher for a flexible question

Use this optional example for a structured question that does not have a fixed query. It uses the same workshop credentials as every other cell, along with the shared model, pinned schema, and configured database. It gives each database query a 15-second timeout.

- **Generated query:** The model creates Cypher from the question.
- **Safety check:** `EXPLAIN` plans the query first. The cell runs only queries that the planner marks as read-only.
- **Output:** The cell shows the query, validation result, records, and errors.

Use a read-only Neo4j user in production. The database then enforces read-only access as well.

In [ ]:
TEXT2CYPHER_TIMEOUT_SECONDS = 15


def run_optional_text2cypher(question):
    outcome = {
        'question': question,
        'generated_cypher': '',
        'read_only_validation': 'not planned',
        'records': [],
        'result_count': 0,
        'displayed_count': 0,
        'execution_error': None,
    }
    try:
        prompt = GRAPH_QUERY_PROMPT.format(
            schema=pinned_schema_text(),
            examples=' '.join(GRAPH_QUERY_EXAMPLES),
            query_text=question,
        )
        response = BedrockLLM(region_name=aws_region()).invoke(prompt)
        cypher = extract_cypher(response.content)
        outcome['generated_cypher'] = cypher

        with driver.session(
            database=DATABASE,
            default_access_mode=READ_ACCESS,
        ) as session:
            summary = session.run(
                Query(f'EXPLAIN {cypher}', timeout=TEXT2CYPHER_TIMEOUT_SECONDS)
            ).consume()
            if summary.query_type != 'r':
                outcome['read_only_validation'] = (
                    f'rejected: query_type={summary.query_type}'
                )
                raise RuntimeError('The planner did not classify the query as read-only')
            outcome['read_only_validation'] = 'passed: EXPLAIN query_type=r'
            all_records = session.run(
                Query(cypher, timeout=TEXT2CYPHER_TIMEOUT_SECONDS)
            ).data()
            outcome['result_count'] = len(all_records)
            outcome['records'] = all_records[:25]
            outcome['displayed_count'] = len(outcome['records'])
    except Exception as exc:
        outcome['execution_error'] = f'{type(exc).__name__}: {exc}'
    return outcome


text2cypher_outcome = run_optional_text2cypher(CHICAGO_QUESTION)
for field, value in text2cypher_outcome.items():
    print(f'{field}: {value}')
print('Supporting Text2Cypher output shown. Fixed Cypher remains the acceptance path.')

## Select a retriever for the question

| Question type | Use first | Check these records | Use in this workshop |
|---|---|---|---|
| Question uses different wording | `VectorRetriever` | Ranked `Chunk` nodes, vector scores, source paths | Basic semantic search |
| Question includes an exact term | `HybridRetriever` | Exact-term hits and ranked `Chunk` nodes | Exact-term support |
| Question needs related hotel facts | `VectorCypherRetriever` | Source `Chunk`, graph fields, relationships, source paths | Connected-context search |
| Question has known, fixed conditions | Reviewed fixed Cypher | Candidate, qualifier, and exclusion records | Fixed, repeatable filter |
| Flexible structured question | Text2Cypher | Generated query, planner result, records, errors | Optional query with a safety check |
| Module 3 application question | `search_hotel_knowledge` | Hybrid-Cypher context records | Selected handoff retriever |

- **`HybridCypherRetriever`:** Combines exact-term search with reviewed graph expansion.
- **Module 2 choice:** The booking question needs an exact hotel name and related named fields in one context record.
- **Module 3 handoff:** The application exposes this retriever through `search_hotel_knowledge`. Module 3 can then focus on grounded answers, abstaining when context is missing, and protected reservation writes.

## Choose retrieval chunk sizes

This workshop uses large chunks during graph extraction. Each hotel stays together while the pipeline creates entities and relationships.

- **Extraction chunks:** Keep hotel information together for graph creation.
- **Retrieval chunks:** A production system can create separate, smaller chunks for search.
- **Chunk size:** Choose a size that fits the source material, expected questions, and context budget.

In [ ]:
selected_module_3_retriever = search_hotel_knowledge
assert selected_module_3_retriever.__name__ == 'search_hotel_knowledge', (
    f'Expected Module 3 handoff to search_hotel_knowledge; observed '
    f'{selected_module_3_retriever.__name__}'
)
print('Selected for Module 3: workshop.hybrid_retrieval.search_hotel_knowledge')
driver.close()
print('Connection closed.')